# Q-Learning and Policy Gradients — Exercises

Companion to the note [Q-Learning and Policy Gradients](Q-Learning%20and%20Policy%20Gradients.md).

Practise model-free RL: TD(0), SARSA vs Q-learning, maximisation bias and Double Q-learning, the DQN tricks, REINFORCE with and without a baseline, and the PPO clipped objective. Small tabular environments are implemented in plain NumPy so everything runs in seconds. Foundations: [[RL Basics and MDPs]].

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy import integrate, stats
rng = np.random.default_rng(0)

# --- Corridor: states 0..5, start 0, state 5 terminal. Actions 0 = left, 1 = right.
#     Every step costs -1, the step that reaches state 5 pays +10 instead.
N_COR = 6
def corridor_step(s, a):
    s2 = max(s - 1, 0) if a == 0 else s + 1
    done = s2 == N_COR - 1
    return s2, (10.0 if done else -1.0), done

# --- Cliff walking (Sutton & Barto Example 6.6): 4 x 12 grid, start bottom-left, goal bottom-right.
#     Each step -1; stepping into the cliff (bottom row between start and goal) gives -100 and resets to start.
ROWS, COLS = 4, 12
START, GOAL = (3, 0), (3, 11)
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # up, right, down, left
def cliff_step(s, a):
    r_, c_ = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
    r_, c_ = min(max(r_, 0), ROWS - 1), min(max(c_, 0), COLS - 1)
    if r_ == 3 and 1 <= c_ <= 10:
        return START, -100.0, False
    return (r_, c_), -1.0, (r_, c_) == GOAL

## Part A · Concepts

### Exercise 1 · On-policy vs off-policy targets
*🧠 Concept · ★☆☆*

Compare the update targets of SARSA, $r + \gamma Q(s',a')$, and Q-learning, $r + \gamma\max_{a'}Q(s',a')$.
(a) Which policy's value does each one learn when the agent behaves ε-greedily? (b) Why is Q-learning called *off-policy*?
(c) Why does that make Q-learning compatible with a replay buffer of old transitions while SARSA is not?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

SARSA needs the *next action actually chosen* by the current behaviour policy; Q-learning only needs $(s,a,r,s')$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) SARSA learns $Q^{\pi_\varepsilon}$ of the ε-greedy policy it follows (on-policy). Q-learning learns $Q^*$ of the **greedy** policy, regardless of how it behaves.
(b) Off-policy: the target policy (greedy, via the max) differs from the behaviour policy (ε-greedy) that generated the data.
(c) A transition $(s,a,r,s')$ from any old policy is still a valid sample for the Q-learning target, because the max is evaluated with the *current* $Q$.
SARSA's $a'$ was chosen by an old policy, so the sample would estimate the wrong policy's value. DQN therefore builds on Q-learning.

</details>

### Exercise 2 · Why DQN needs its tricks
*🧠 Concept · ★★☆*

Naively plugging a neural network into Q-learning is unstable. For each DQN ingredient, say which problem it fixes:
(a) experience replay, (b) a target network $Q_{\theta^-}$ updated every $C$ steps, (c) Double DQN, (d) Dueling heads $Q = V + A - \bar A$,
(e) prioritized replay. What is the "deadly triad"?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about correlated consecutive samples, a moving regression target, and the $\max$ over noisy estimates.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Replay** breaks the temporal correlation of consecutive samples (SGD assumes i.i.d. data) and reuses each transition many times.
(b) **Target network**: the TD target $r + \gamma\max_{a'}Q_{\theta^-}(s',a')$ is frozen for $C$ steps, so the network is not chasing a target that moves with every gradient step.
(c) **Double DQN** selects $a' = \arg\max Q_\theta(s',\cdot)$ with the online net but evaluates it with $Q_{\theta^-}$, removing the maximisation bias (Exercise 10).
(d) **Dueling** separates "how good is this state" from "how much better is this action", so $V$ is learned from every transition, helpful when actions matter little.
(e) **Prioritized replay** samples transitions with large TD error more often (with importance weights to correct the bias).

**Deadly triad** (Sutton & Barto): function approximation + bootstrapping + off-policy learning together can diverge. DQN has all three; its tricks mitigate it.

</details>

### Exercise 3 · Choosing an algorithm
*🧠 Concept · ★★☆*

Recommend an algorithm family from the note (DQN-style, REINFORCE, PPO, SAC/TD3) for each setting and justify briefly:
1. Atari game, discrete actions, a simulator that is cheap to run.
2. A robot arm with 7 continuous joint torques, real-world samples are expensive.
3. Fine-tuning a language model from a learned reward model (RLHF).
4. A teaching example with a 3-armed bandit and a softmax policy.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Value-based methods need $\max_a Q(s,a)$, which is hard over continuous actions. Off-policy methods reuse data (sample efficient).

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **DQN** (or Rainbow): discrete actions make $\max_a$ trivial; PPO also works well.
2. **SAC/TD3**: off-policy actor-critics for continuous actions; the replay buffer makes them far more sample-efficient than on-policy PPO.
3. **PPO**: the action space is the vocabulary (huge), the policy is the LM itself; PPO's clipped objective keeps the policy close to the previous one (plus a KL penalty to the reference model).
4. **REINFORCE**: simplest unbiased policy gradient, ideal to see the score-function estimator and the effect of a baseline (Exercises 15–16).

</details>

### Exercise 4 · Monte Carlo vs TD: bias and variance
*🧠 Concept · ★★☆*

The MC target is the full return $G_t$; the TD(0) target is $r_{t+1} + \gamma V(s_{t+1})$.
Which is biased, which has higher variance, and why? Which one can learn from incomplete episodes / continuing tasks?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$G_t$ depends on *all* future random rewards and actions; the TD target depends on one step plus the current (wrong) estimate $V$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **MC**: $\mathbb E[G_t\mid s_t] = V^\pi(s_t)$, so the target is **unbiased**, but it sums many random rewards → **high variance**. It needs the episode to end.
- **TD(0)**: uses the estimate $V(s_{t+1})$ (bootstrapping), so it is **biased** while $V\neq V^\pi$, but depends on only one random transition → **low variance**.
  It learns online after every step and works for continuing tasks.

n-step returns and TD($\lambda$) interpolate between the two; actor-critic methods use TD-style critics for the same variance reason.

</details>

## Part B · By hand

### Exercise 5 · One TD(0) update
*✍️ By hand · ★☆☆*

$V(s) = 2$, $V(s') = 3$. The agent observes $r = 1$ while moving from $s$ to $s'$. With $\alpha = 0.1$ and $\gamma = 0.9$,
compute the TD error $\delta$ and the updated $V(s)$.

In [ ]:
delta = None
V_s_new = None

check('TD error', delta, 1 + 0.9*3 - 2)
check('new V(s)', V_s_new, 2 + 0.1*(1 + 0.9*3 - 2))

<details>
<summary><b>💡 Hint</b></summary>

$\delta = r + \gamma V(s') - V(s)$, then $V(s)\leftarrow V(s) + \alpha\delta$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\delta = 1 + 0.9\cdot 3 - 2 = 1.7$; $V(s) \leftarrow 2 + 0.1\cdot1.7 = 2.17$.

```python
delta = 1.7
V_s_new = 2.17
```

</details>

### Exercise 6 · SARSA vs Q-learning on the same transition
*✍️ By hand · ★☆☆*

$Q(s,a) = 1$, reward $r = 0$, $\gamma = 0.9$, $\alpha=0.5$. In $s'$ the estimates are $Q(s',a_0) = 2$, $Q(s',a_1) = 5$.
The ε-greedy behaviour policy happened to pick the exploratory action $a' = a_0$ in $s'$.
Compute the new $Q(s,a)$ under SARSA and under Q-learning.

In [ ]:
Q_sarsa = None
Q_qlearn = None

check('SARSA', Q_sarsa, 1 + 0.5*(0 + 0.9*2 - 1))
check('Q-learning', Q_qlearn, 1 + 0.5*(0 + 0.9*max(2, 5) - 1))

<details>
<summary><b>💡 Hint</b></summary>

SARSA uses $Q(s', a')$ with the action actually taken; Q-learning uses $\max_{a'}Q(s',a')$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SARSA: target $= 0 + 0.9\cdot 2 = 1.8$, $Q \leftarrow 1 + 0.5(1.8 - 1) = 1.4$.
Q-learning: target $= 0 + 0.9\cdot 5 = 4.5$, $Q\leftarrow 1 + 0.5(4.5-1) = 2.75$.
SARSA "pays" for the exploratory move; Q-learning ignores it. This is exactly why SARSA learns a safer path on the cliff (Exercise 13).

```python
Q_sarsa = 1.4
Q_qlearn = 2.75
```

</details>

### Exercise 7 · Softmax score function and a REINFORCE step
*✍️ By hand · ★★☆*

A softmax policy over two actions has logits $\theta = (\theta_0,\theta_1)$, $\pi_\theta(a) = e^{\theta_a}/\sum_b e^{\theta_b}$.

(a) Show that $\nabla_\theta\log\pi_\theta(a) = e_a - \pi_\theta$ (one-hot minus probabilities).
(b) With $\theta = (0, \ln 3)$ the agent takes $a = 0$ and receives return $G = 2$. Perform one REINFORCE step
$\theta \leftarrow \theta + \alpha\, G\,\nabla_\theta\log\pi_\theta(a)$ with $\alpha = 0.1$.

In [ ]:
pi_theta = None   # [pi(0), pi(1)]
theta_new = None  # [theta0, theta1]

_th = np.array([0, np.log(3)]); _p = np.exp(_th) / np.exp(_th).sum()
# reference gradient by finite differences of log pi(0)
_g = np.array([(np.log(np.exp(_th + e)[0] / np.exp(_th + e).sum()) - np.log(_p[0])) / 1e-7 for e in 1e-7 * np.eye(2)])
check('pi', pi_theta, _p)
check('theta after update', theta_new, _th + 0.1 * 2 * _g, tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

$\log\pi_\theta(a) = \theta_a - \log\sum_b e^{\theta_b}$. Differentiate w.r.t. $\theta_k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\partial_{\theta_k}[\theta_a - \log\sum_b e^{\theta_b}] = \mathbb 1[k=a] - \frac{e^{\theta_k}}{\sum_b e^{\theta_b}} = \mathbb 1[k=a] - \pi_\theta(k)$.

(b) $\pi = (1, 3)/4 = (0.25, 0.75)$. Gradient for $a=0$: $(1-0.25,\ 0-0.75) = (0.75, -0.75)$.
$\theta \leftarrow (0, \ln 3) + 0.1\cdot 2\cdot(0.75,-0.75) = (0.15,\ \ln 3 - 0.15) \approx (0.15, 0.9486)$.
The probability of the rewarded action rises (from 0.25 to $\approx 0.31$). Note: if $G$ were negative, it would fall: that is why the *sign* of $G - b$ matters.

```python
pi_theta = [0.25, 0.75]
theta_new = [0.15, np.log(3) - 0.15]
```

</details>

### Exercise 8 · A baseline does not bias the gradient
*✍️ By hand · ★★☆*

Show that for any baseline $b(s)$ that does not depend on the action,
$$\mathbb E_{a\sim\pi_\theta(\cdot\mid s)}\big[\nabla_\theta\log\pi_\theta(a\mid s)\, b(s)\big] = 0.$$
Conclude that $\mathbb E[\nabla\log\pi\,(G_t - b)] = \mathbb E[\nabla\log\pi\, G_t]$. Which choice of $b$ is popular, and what is $G_t - b(s_t)$ then called?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write the expectation as a sum $\sum_a \pi_\theta(a\mid s)\nabla_\theta\log\pi_\theta(a\mid s)$ and use $\pi\nabla\log\pi = \nabla\pi$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sum_a \pi_\theta(a\mid s)\nabla_\theta\log\pi_\theta(a\mid s)\,b(s) = b(s)\sum_a\nabla_\theta\pi_\theta(a\mid s) = b(s)\nabla_\theta\underbrace{\sum_a\pi_\theta(a\mid s)}_{=1} = 0$.

So subtracting $b$ leaves the expected gradient unchanged while it can greatly reduce variance (Exercise 16).
The popular choice is $b(s) = V(s)$ learned by a critic; $A_t = G_t - V(s_t)$ (or $r + \gamma V(s') - V(s)$) is the **advantage**, which gives actor–critic methods (A2C, PPO).

</details>

### Exercise 9 · The PPO clipped objective
*✍️ By hand · ★★☆*

PPO maximises $L^{\text{CLIP}} = \mathbb E\big[\min\big(\rho\,A,\ \text{clip}(\rho, 1-\epsilon, 1+\epsilon)\,A\big)\big]$ with
$\rho = \pi_\theta(a\mid s)/\pi_{\theta_{\text{old}}}(a\mid s)$. With $\epsilon = 0.2$ compute the per-sample objective for

(i) $\rho = 1.5,\ A = 2$; (ii) $\rho = 0.5,\ A = -1$; (iii) $\rho = 0.7,\ A = 1$.

In which cases is the gradient w.r.t. $\theta$ zero? Explain what the clipping achieves.

In [ ]:
L_clip = None  # [case i, case ii, case iii]

check('L_clip', L_clip, [min(r*A, np.clip(r, 0.8, 1.2)*A) for r, A in [(1.5, 2), (0.5, -1), (0.7, 1)]])

<details>
<summary><b>💡 Hint</b></summary>

Compute both terms and take the smaller one. If the clipped term wins, the objective is constant in $\rho$ locally.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) $\min(3,\ 1.2\cdot2 = 2.4) = 2.4$ → clipped, **zero gradient**: the action is already much more likely than before, no further push.
(ii) $\min(-0.5,\ 0.8\cdot(-1) = -0.8) = -0.8$ → clipped, **zero gradient**: a bad action has already been made much less likely.
(iii) $\min(0.7,\ 0.8) = 0.7$ → unclipped, gradient $\neq 0$: a good action became *less* likely, so PPO still corrects it (the min makes the objective a pessimistic lower bound).

Clipping removes the incentive to move $\rho$ outside $[1-\epsilon, 1+\epsilon]$ in the direction that improves the objective, a cheap first-order
stand-in for TRPO's KL trust region. That lets PPO take several epochs of minibatch SGD on the same batch safely.

```python
L_clip = [2.4, -0.8, 0.7]
```

</details>

### Exercise 10 · Maximisation bias
*✍️ By hand · ★★★*

In state $s'$ two actions both have true value $Q(s',a) = 0$, but our estimates are $\hat Q(s',a) = \varepsilon_a$ with
$\varepsilon_1,\varepsilon_2 \stackrel{iid}{\sim}\mathcal N(0,1)$. Compute $\mathbb E[\max_a\hat Q(s',a)]$ exactly.
Why does this bias Q-learning's target upwards, and how does Double Q-learning avoid it?

In [ ]:
E_max = None

check('E[max]', E_max, integrate.quad(lambda x: x * 2 * stats.norm.pdf(x) * stats.norm.cdf(x), -np.inf, np.inf)[0], tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

$\max(X_1,X_2) = \frac{X_1+X_2}{2} + \frac{|X_1-X_2|}{2}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$X_1-X_2\sim\mathcal N(0,2)$ and $\mathbb E|Z| = \sigma\sqrt{2/\pi}$ for $Z\sim\mathcal N(0,\sigma^2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[\max] = 0 + \tfrac12\mathbb E|X_1-X_2| = \tfrac12\sqrt2\sqrt{2/\pi} = 1/\sqrt\pi \approx 0.564$ although the true max is $0$.

Q-learning uses $\max_{a'}\hat Q$ both to **select** and to **evaluate** the action, so noise that happens to be positive is selected:
$\mathbb E[\max_a\hat Q]\ge\max_a\mathbb E[\hat Q]$ (Jensen). The bias propagates through bootstrapping.
Double Q-learning keeps two independent estimates: $a^* = \arg\max_a Q_1(s',a)$, target $r + \gamma Q_2(s',a^*)$. Since $Q_2$'s noise is independent of the selection,
$\mathbb E[Q_2(s',a^*)] = Q(s',a^*)$, which is unbiased (in fact slightly pessimistic).

```python
E_max = 1 / np.sqrt(np.pi)
```

</details>

## Part C · Tabular value-based methods

### Exercise 11 · TD(0) on the random walk
*💻 Code · ★☆☆*

Sutton & Barto's random walk: non-terminal states A–E (indices 1..5), terminals 0 and 6, start in C (index 3), each step moves left or right
with probability ½. Reward is $+1$ on reaching the right terminal, else 0; $\gamma=1$. The true values are $V(A..E) = (1,2,3,4,5)/6$.

Implement TD(0) with $\alpha = 0.02$ for 3000 episodes (init $V = 0.5$ for non-terminals, $0$ for terminals). Store $V$ for A–E in `V_td`.

In [ ]:
def td0_random_walk(n_episodes=3000, alpha=0.02, seed=0):
    r = np.random.default_rng(seed)
    V = np.r_[0, np.full(5, 0.5), 0]
    # TODO: for each episode: s = 3; until terminal: s2 = s +/- 1; reward = 1 if s2 == 6 else 0; TD update
    return None

V_td = td0_random_walk()

check('TD(0) values', V_td, np.arange(1, 6) / 6, tol=0.08)

<details>
<summary><b>💡 Hint</b></summary>

Terminal values stay 0, so the update at the final step is $V(s) \leftarrow V(s) + \alpha[r - V(s)]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With a constant step size the estimates keep fluctuating around the true values (the error does not go to 0); a decaying $\alpha_t$
with $\sum\alpha_t=\infty$, $\sum\alpha_t^2<\infty$ would make them converge.

```python
def td0_random_walk(n_episodes=3000, alpha=0.02, seed=0):
    r = np.random.default_rng(seed)
    V = np.r_[0, np.full(5, 0.5), 0]
    for _ in range(n_episodes):
        s = 3
        while s not in (0, 6):
            s2 = s + (1 if r.random() < 0.5 else -1)
            rew = 1.0 if s2 == 6 else 0.0
            V[s] += alpha * (rew + V[s2] - V[s])
            s = s2
    return V[1:6]

V_td = td0_random_walk()
print(np.round(V_td, 3))
```

</details>

### Exercise 12 · Q-learning on the corridor
*💻 Code · ★★☆*

Use `corridor_step(s, a)` from the setup (6 states, start 0, terminal 5, step reward −1, +10 on arrival).
Implement tabular Q-learning with ε-greedy behaviour ($\varepsilon=0.2$, $\alpha=0.5$, $\gamma=0.9$, 300 episodes, $Q=0$ initially).
Store the learned table in `Q_cor` (shape `(6, 2)`). Compare with $Q^*$ computed by value iteration on the known model.

In [ ]:
def q_learning_corridor(n_episodes=300, eps=0.2, alpha=0.5, gamma=0.9, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros((N_COR, 2))
    # TODO
    return None

Q_cor = q_learning_corridor()

_Q = np.zeros((N_COR, 2))
for _ in range(500):  # value iteration on the (known, deterministic) model
    for s in range(N_COR - 1):
        for a in range(2):
            s2, rew, done = corridor_step(s, a)
            _Q[s, a] = rew + (0 if done else 0.9 * _Q[s2].max())
check('Q-learning reaches Q*', Q_cor, _Q, tol=0.05)

<details>
<summary><b>💡 Hint</b></summary>

Break ties randomly in the greedy step (`r.choice(np.flatnonzero(Q[s] == Q[s].max()))`); the target for a terminal transition is just $r$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$Q^*(s,\text{right})$ is $10, 8, 6.2, 4.58, 3.122$ for $s = 4,3,2,1,0$, i.e. $V^*(s) = -1 + 0.9V^*(s+1)$. In a deterministic environment
with $\alpha = 0.5$ and every $(s,a)$ visited often, Q-learning converges to $Q^*$ even though it behaves ε-greedily: it is off-policy.

```python
def q_learning_corridor(n_episodes=300, eps=0.2, alpha=0.5, gamma=0.9, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros((N_COR, 2))
    for _ in range(n_episodes):
        s, done = 0, False
        while not done:
            a = r.integers(2) if r.random() < eps else r.choice(np.flatnonzero(Q[s] == Q[s].max()))
            s2, rew, done = corridor_step(s, a)
            target = rew if done else rew + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
    return Q

Q_cor = q_learning_corridor()
print(np.round(Q_cor, 3))
```

</details>

### Exercise 13 · SARSA vs Q-learning on the cliff
*💻 Code · ★★★*

Reproduce Sutton & Barto's cliff-walking result with `cliff_step` from the setup ($\varepsilon=0.1$, $\alpha=0.5$, $\gamma=1$, 500 episodes).
Implement `train(method)` for `method in {'sarsa', 'qlearning'}` returning the per-episode return (sum of rewards **while training**)
and the final Q table (shape `(4, 12, 4)`). Store the mean training return over the last 100 episodes in `avg_ret = {'sarsa': ..., 'qlearning': ...}`
and the length of the **greedy** path learned by Q-learning in `greedy_len_q`.

In [ ]:
def train(method, n_episodes=500, eps=0.1, alpha=0.5, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros((ROWS, COLS, 4))
    returns = []
    # TODO
    return None, None

avg_ret = None       # {'sarsa': ..., 'qlearning': ...}
greedy_len_q = None  # number of steps of the greedy Q-learning path from START to GOAL

if avg_ret is not None:
    check('SARSA earns more while exploring', avg_ret['sarsa'] > avg_ret['qlearning'], True)
else:
    check('SARSA earns more while exploring', None, True)
check('Q-learning greedy path is the shortest (along the cliff)', greedy_len_q, 13)

<details>
<summary><b>💡 Hint 1</b></summary>

Write a helper `eps_greedy(Q, s)` and use it for both methods; the only difference is the target ($Q(s',a')$ vs $\max Q(s',\cdot)$).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For the greedy path: start at `START`, follow `argmax` for at most 100 steps and count steps until `GOAL`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Q-learning learns the optimal path right along the cliff edge (13 steps, return −13), but while exploring with ε = 0.1 it
regularly falls off, so its *online* return is worse (≈ −62 over the last 100 episodes here). SARSA evaluates the ε-greedy policy it actually follows, so it learns the
safer path further from the edge and earns more during training (≈ −26). As $\varepsilon\to0$ both converge to the optimal path.

```python
def eps_greedy(Q, s, eps, r):
    if r.random() < eps:
        return r.integers(4)
    q = Q[s]
    return r.choice(np.flatnonzero(q == q.max()))

def train(method, n_episodes=500, eps=0.1, alpha=0.5, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros((ROWS, COLS, 4))
    returns = []
    for _ in range(n_episodes):
        s, done, G = START, False, 0.0
        a = eps_greedy(Q, s, eps, r)
        while not done:
            s2, rew, done = cliff_step(s, a)
            G += rew
            a2 = eps_greedy(Q, s2, eps, r)
            nxt = Q[s2][a2] if method == 'sarsa' else Q[s2].max()
            Q[s][a] += alpha * (rew + (0 if done else nxt) - Q[s][a])
            s, a = s2, a2
        returns.append(G)
    return np.array(returns), Q

res = {m: train(m) for m in ('sarsa', 'qlearning')}
avg_ret = {m: res[m][0][-100:].mean() for m in res}
print(avg_ret)

s, greedy_len_q = START, 0
Qq = res['qlearning'][1]
while s != GOAL and greedy_len_q < 100:
    s, _, _ = cliff_step(s, int(np.argmax(Qq[s])))
    greedy_len_q += 1
print('greedy path length (Q-learning):', greedy_len_q)
```

</details>

### Exercise 14 · Double Q-learning fixes maximisation bias
*💻 Code · ★★★*

Sutton & Barto Example 6.7: from start state $A$, action *right* ends the episode with reward 0; action *left* goes to state $B$ (reward 0),
where each of 10 actions ends the episode with reward $\sim\mathcal N(-0.1, 1)$. So *left* is worse in expectation.

Implement Q-learning and Double Q-learning ($\varepsilon = 0.1$, $\alpha = 0.1$, $\gamma = 1$) and record, for each of the first 300 episodes,
whether the agent chose *left* in $A$. Average over 100 runs and store the overall fraction of *left* choices in
`left_frac = {'q': ..., 'double': ...}`. Relate the result to Exercise 10.

In [ ]:
def run_ab(double, n_episodes=300, seed=0):
    r = np.random.default_rng(seed)
    # Q tables: QA has 2 actions (0 = left, 1 = right), QB has 10 actions. For double, keep two copies of each.
    # TODO: return a boolean array: did the agent go left in episode i?
    return None

left_frac = None

if left_frac is not None:
    check('Double Q goes left much less often', left_frac['double'] < left_frac['q'] - 0.1, True)
else:
    check('Double Q goes left much less often', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Double Q-learning: with prob. ½ update $Q_1$ using target $r + \gamma Q_2(s', \arg\max_{a'} Q_1(s',a'))$, otherwise swap roles. Act ε-greedily on $Q_1 + Q_2$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Break ties randomly; otherwise `argmax` always picks action 0 = left at the start.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Q-learning picks *left* far more often than the ε-greedy minimum of 5% in early episodes: after a few visits to $B$, $\max_b Q(B,b)$ is positive
by chance (Exercise 10), and that optimistic value is bootstrapped into $Q(A,\text{left})$. Double Q-learning decouples selection and evaluation,
so its estimate of $Q(A,\text{left})$ is not inflated and it quickly approaches the 5% floor. Here: Q-learning ≈ 40% *left* over the first 300 episodes, Double Q ≈ 11%.

```python
def _greedy(q, r):
    return r.choice(np.flatnonzero(q == q.max()))

def run_ab(double, n_episodes=300, seed=0):
    r = np.random.default_rng(seed)
    QA = np.zeros((2, 2)); QB = np.zeros((2, 10))  # index 0/1 = the two copies (only copy 0 used without double)
    went_left = np.zeros(n_episodes, bool)
    for ep in range(n_episodes):
        qa = QA.sum(0) if double else QA[0]
        a = r.integers(2) if r.random() < 0.1 else _greedy(qa, r)
        if a == 1:  # right: terminal, reward 0
            i = r.integers(2) if double else 0
            QA[i, 1] += 0.1 * (0 - QA[i, 1])
            continue
        went_left[ep] = True
        i = r.integers(2) if double else 0
        j = 1 - i if double else 0
        # A --left--> B, reward 0: target = Q_j(B, argmax_b Q_i(B, b))
        QA[i, 0] += 0.1 * (QB[j, _greedy(QB[i], r)] - QA[i, 0])
        qb = QB.sum(0) if double else QB[0]
        b = r.integers(10) if r.random() < 0.1 else _greedy(qb, r)
        k = r.integers(2) if double else 0
        QB[k, b] += 0.1 * (r.normal(-0.1, 1) - QB[k, b])
    return went_left

left_frac = {name: np.mean([run_ab(d, seed=s).mean() for s in range(100)]) for name, d in (('q', False), ('double', True))}
print(left_frac)
```

</details>

## Part D · Policy gradients

### Exercise 15 · REINFORCE on a bandit
*💻 Code · ★★☆*

A 3-armed bandit pays rewards $\sim\mathcal N(\mu_a, 1)$ with $\mu = (1, 2, 1.5)$. Train a softmax policy with logits $\theta\in\mathbb R^3$ (init 0)
by REINFORCE, $\theta\leftarrow\theta + \alpha\,(r - b)\,(e_a - \pi_\theta)$, with $\alpha = 0.05$ for 2000 steps, where the baseline $b$ is a running
average of past rewards (`b += 0.05 * (r - b)`). Store the final $\pi_\theta$ in `pi_final`.

In [ ]:
mu_arms = np.array([1.0, 2.0, 1.5])
def softmax(z):
    z = z - z.max()
    return np.exp(z) / np.exp(z).sum()

def reinforce_bandit(steps=2000, alpha=0.05, use_baseline=True, seed=0):
    r = np.random.default_rng(seed)
    theta = np.zeros(3)
    # TODO
    return None

pi_final = reinforce_bandit()

if pi_final is not None:
    check('valid distribution', np.sum(pi_final), 1.0)
    check('policy concentrates on the best arm', pi_final[np.argmax(mu_arms)] > 0.9, True)
else:
    check('policy concentrates on the best arm', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Sample `a = r.choice(3, p=pi)`, reward `r.normal(mu_arms[a], 1)`. The score function is `np.eye(3)[a] - pi` (Exercise 7).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The policy gradient for a bandit is $\nabla J = \mathbb E[(r - b)\nabla\log\pi(a)]$; one sample gives an unbiased stochastic estimate.
With rewards all positive and no baseline, *every* sampled action is pushed up (just by different amounts), which makes learning noisy; the baseline
turns that into "better/worse than usual". Try `use_baseline=False` and compare the trajectory of $\pi_\theta$.

```python
mu_arms = np.array([1.0, 2.0, 1.5])
def softmax(z):
    z = z - z.max()
    return np.exp(z) / np.exp(z).sum()

def reinforce_bandit(steps=2000, alpha=0.05, use_baseline=True, seed=0):
    r = np.random.default_rng(seed)
    theta, b = np.zeros(3), 0.0
    for _ in range(steps):
        pi = softmax(theta)
        a = r.choice(3, p=pi)
        rew = r.normal(mu_arms[a], 1)
        adv = rew - b if use_baseline else rew
        theta += alpha * adv * (np.eye(3)[a] - pi)
        if use_baseline:
            b += 0.05 * (rew - b)
    return softmax(theta)

pi_final = reinforce_bandit()
print(np.round(pi_final, 3))
```

</details>

### Exercise 16 · Measuring the variance reduction of a baseline
*💻 Code · ★★☆*

Fix the uniform policy $\theta = 0$ on the bandit above. Draw 100 000 single-sample gradient estimates $g = (r - b)(e_a - \pi)$
for $b = 0$ and for $b = \mathbb E_\pi[r] = 1.5$. Store the empirical means (`g_mean0`, `g_mean_b`, each shape `(3,)`) and the total
variances $\operatorname{tr}\operatorname{Cov}(g)$ (`var0`, `var_b`). Compare the means with the exact gradient
$\nabla J = \sum_a \pi(a)\mu_a(e_a - \pi)$.

In [ ]:
var_rng = np.random.default_rng(3)
g_mean0 = g_mean_b = None
var0 = var_b = None

_pi = np.full(3, 1/3)
_grad = sum(_pi[a] * mu_arms[a] * (np.eye(3)[a] - _pi) for a in range(3))
check('mean without baseline is unbiased', g_mean0, _grad, tol=0.02)
check('mean with baseline is unbiased', g_mean_b, _grad, tol=0.02)
check('baseline reduces variance', None if var0 is None else var_b < var0, True)

<details>
<summary><b>💡 Hint</b></summary>

Vectorise: `a = var_rng.integers(3, size=n)`, `rew = var_rng.normal(mu_arms[a], 1)`, `G = (rew - b)[:, None] * (np.eye(3)[a] - 1/3)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both estimators have the same mean $\nabla J = (-1/6,\ 1/6,\ 0)$ (unbiased, Exercise 8), but the total variance drops from about $2.23$ to about $0.73$
(roughly 3× fewer samples for the same accuracy). The optimal constant baseline is close to $\mathbb E[r]$ (weighted by $\lVert\nabla\log\pi\rVert^2$).

```python
var_rng = np.random.default_rng(3)
n = 100_000
a = var_rng.integers(3, size=n)
rew = var_rng.normal(mu_arms[a], 1)
score = np.eye(3)[a] - 1 / 3
G0 = rew[:, None] * score
Gb = (rew - 1.5)[:, None] * score
g_mean0, g_mean_b = G0.mean(0), Gb.mean(0)
var0, var_b = np.trace(np.cov(G0.T)), np.trace(np.cov(Gb.T))
print(np.round(g_mean0, 3), np.round(g_mean_b, 3), round(var0, 3), round(var_b, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Q-Learning and Policy Gradients](Q-Learning%20and%20Policy%20Gradients.md).*